<a href="https://colab.research.google.com/github/Geauga/Proteina-Complexa-dev/blob/main/coLab/Proteina_Complexa_coLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  [No need to re-run after session restart] Upload Custom PDB

<details>
<summary><font size="5"><b>⚙️ Click to Expand: Global Configuration & Workflow Instructions</b></font></summary>

<br>

### 💡 OVERVIEW
This notebook utilizes a professional **"Front-end UI / Back-end Engine"** architecture. Please configure your parameters in the UI cells below, then sequentially execute the hidden engines in the collapsed sections.

### 🛑 CRITICAL NOTICE: MANDATORY KERNEL RESTART
After completing **Section 1** (Environment Setup), you MUST restart the Colab runtime to apply core library upgrades (JAX/Flax/CUDA).
**ACTION:** Click `Runtime -> Restart session` before running Section 2. Failure to do so will cause the pipeline to crash.

### 📦 ZERO-CONFIGURATION ASSETS
The demo structure (`GFP_1_10.pdb`) is pre-integrated. For custom research, simply upload your own PDB files directly into the `Proteina-Complexa/targets/` directory in your Google Drive.

---
### 🔄 EXECUTION WORKFLOWS

**Workflow A: Standard Pipeline (Linear Generation)**
1. **Global Config:** Adjust parameters in the UI cells below (Cell 2, 4, 5) and run them to register variables.
2. **Setup Env:** Expand [Section 1] and run the hidden setup engines.
3. ⚠️ **RESTART SESSION** (`Runtime -> Restart session`).
4. **Execute Pipeline:** Expand [Section 2] & [Section 3] sequentially and run the hidden engines (Cell 4b, 5b) to start generation.

**Workflow B: Unattended Auto-Pilot (High-Throughput)**
1. Complete Environment Setup [Section 1] & **Restart Session**.
2. **Global Config:** Adjust parameters in Cell 4 (Target) and Cell 9 (Auto-Pilot). Run them.
3. **Start Auto-Pilot:** Expand [Section 2] to prepare the target, then expand [Section 3] and run the hidden **Cell 9b** engine. The system will continuously generate and log until the target Hit count is reached.

</details>

In [18]:

# @markdown Complete this dashboard **ONCE**. All settings are persistently saved to your Google Drive. You can safely restart the runtime later without losing any data.

import os, json, shutil
from pathlib import Path
from IPython.display import display, HTML

# ==========================================
# 0. Infrastructure & Google Drive Mount
# ==========================================
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    print("🔄 Mounting Google Drive for persistent storage...")
    drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGETS_DIR = BASE_DIR / 'targets'
TARGETS_DIR.mkdir(parents=True, exist_ok=True)
SETTING_FILE = BASE_DIR / 'internal_settings.json'

# ==========================================
# --- Configuration for Environment Setup & Cache ---
# ==========================================
# @markdown ### 💾 Environment & Cache Settings
# @markdown Check the box below to back up compiled `uv` dependencies and C++ build cache to Google Drive.
# @markdown **⚠️ CRITICAL STORAGE WARNING:** Requires **5GB** free space.
enable_cache_backup = True # @param {type:"boolean"}

# ==========================================
# --- Configuration for Target Setup & Processing ---
# ==========================================
# @markdown ---
# @markdown ### 📁 Target File Selection
# @markdown Check to upload a new PDB from your computer, or leave unchecked to use the default/existing filename.

# @markdown <br>
# @markdown <font size="4" color="#d93f0b"><b>👉 UPLOAD NEW PDB (Check to upload) :</b></font>
UPLOAD_NEW_PDB = False #@param {type:"boolean"}
pdb_file_name = "GFP_1_10.pdb" #@param {type:"string"}

# @markdown ### 🧬 Target Definition
task_name = "GFP_1_10" #@param {type:"string"}
target_chains = "A" #@param {type:"string"}

# @markdown ### 🎯 Interaction Hotspots (Comma separated)
hotspots_input = "40,42,44,61,64,65,68,71" #@param {type:"string"}

# @markdown ### 📏 Binder Peptide Length
binder_length_min = 10 #@param {type:"integer"}
binder_length_max = 20 #@param {type:"integer"}

# ==========================================
# --- Configuration for Standard Generation Pipeline ---
# ==========================================
# @markdown ---
# @markdown ### 🧬 Generation Settings
num_designs = 8 #@param {type:"integer"}
gen_batch_size = 8 #@param {type:"slider", min:1, max:64, step:1}

# @markdown ### ⚙️ Pipeline Toggles
run_filter = True #@param {type:"boolean"}
run_evaluate = True #@param {type:"boolean"}
run_analyze = True #@param {type:"boolean"}

# ==========================================
# --- Configuration for Auto-Pilot High-Throughput ---
# ==========================================
# @markdown ---
# @markdown ### 🤖 Auto-Pilot & Screening Criteria
target_threshold = 0.80 #@param {type:"number"}
target_success_count = 10 #@param {type:"integer"}

# @markdown ### 🔄 Cycle Execution Parameters
designs_per_loop = 24 #@param {type:"integer"}
max_iterations = 50 #@param {type:"integer"}

# @markdown ### ⚡ VRAM Optimization (Batch Size)
batch_size = 8 #@param {type:"slider", min:1, max:64, step:1}


# ==========================================
# Execution Logic: Upload & Data Persistence
# ==========================================

# 1. Handle Interactive PDB Upload
if UPLOAD_NEW_PDB:
    from google.colab import files
    display(HTML('<p style="font-size:16px; font-weight:bold; color:#E32636;">👉 PLEASE CLICK "CHOOSE FILES" BELOW TO UPLOAD YOUR PDB:</p>'))
    uploaded = files.upload()
    if uploaded:
        for filename in uploaded.keys():
            shutil.move(filename, TARGETS_DIR / filename)
            pdb_file_name = filename  # Override text parameter with the actual uploaded filename
            print(f"✅ Successfully uploaded and set target to: {pdb_file_name}")
            break
    else:
        print("⚠️ Upload cancelled. Falling back to the text input filename.")
else:
    print(f"📁 Target set to: {pdb_file_name}")

# 2. Bundle all global variables into a dictionary
settings = {
    # Environment Cache
    "enable_cache_backup": enable_cache_backup,

    # Target Configuration
    "pdb_file": pdb_file_name,
    "task_name": task_name,
    "target_chains": target_chains,
    "hotspots_input": hotspots_input,
    "binder_length_min": binder_length_min,
    "binder_length_max": binder_length_max,

    # Standard Pipeline
    "num_designs": num_designs,
    "gen_batch_size": gen_batch_size,
    "run_filter": run_filter,
    "run_evaluate": run_evaluate,
    "run_analyze": run_analyze,

    # Auto-Pilot Parameters
    "target_threshold": target_threshold,
    "target_success_count": target_success_count,
    "designs_per_loop": designs_per_loop,
    "max_iterations": max_iterations,
    "batch_size": batch_size
}

# 3. Write securely to Google Drive (Survives Kernel Restarts)
with open(SETTING_FILE, 'w') as f:
    json.dump(settings, f, indent=4)

print(f"\n🎯 Target [{task_name}] registered successfully!")
print(f"🎯 Hotspots defined: [{hotspots_input}]")
print("\n💾 SUCCESS: All configuration parameters have been safely backed up to your Google Drive.")
print("🚀 You can now proceed to install the environment and safely click 'Restart Session'. Your settings will NOT be lost!")

📁 Target set to: GFP_1_10.pdb

🎯 Target [GFP_1_10] registered successfully!
🎯 Hotspots defined: [40,42,44,61,64,65,68,71]

💾 SUCCESS: All configuration parameters have been safely backed up to your Google Drive.
🚀 You can now proceed to install the environment and safely click 'Restart Session'. Your settings will NOT be lost!


# Section 1: Environment Configuration and Dependency Management (Restart Session after All 4 Cells are Finished after the First Run)

In [2]:
# Cell_1_Environment_Initialization.py
# Requirement: Upgrade essential logging and serialization dependencies (wandb, protobuf) for the pipeline.

# ==========================================
# --- Environment Base Dependencies ---
# ==========================================
!pip install --upgrade wandb protobuf

# Purpose: Ensure the base environment has the latest required versions of Weights & Biases (wandb) for experiment tracking and protocol buffers (protobuf) for model serialization.
# Upstream Code: The first cell of the provided Colab notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 08:30 EDT.
# Changed Lines:
# * Designated the cell as "Cell 1" in the header.
# * Replaced the informal hash-block comments with professional English documentation and structural dividers.
# * Appended the standard metadata tracking block.

  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 102.2 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.23.1
    Uninstalling wandb-0.23.1:
      Successfully uninstalled wandb-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
proteinfoundation 1.0.0 requires scipy==1.12.0, but you have scipy 1.17.1 which is incompatible.
proteinfoundation 1.0.0 requires wandb==0.23.1, but you have wandb 0.25.1 which is incompatible.


In [3]:
# Cell_2_Mount_Google_Drive.py
# Requirement: Mount Google Drive to the Colab instance to enable access to persistent storage for the Proteina-Complexa project files.

from google.colab import drive
drive.mount('/content/drive')

# Purpose: Connect the Google Colab virtual machine to the user's Google Drive, allowing the pipeline to read configurations, load models, and save generated protein structures persistently.
# Upstream Code: The original Google Drive mounting cell (formerly labeled as Cell 1) from the uploaded notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 08:35 EDT.
# Changed Lines:
# * Designated the cell as "Cell 2" in the header.
# * Appended the standard metadata tracking block.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Cell_2b_environment_setup_engine.py
import os

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

# --- Source Repository Detection and Cloning ---
if not os.path.exists(ROOT_DIR):
    print(f">>> Source repository not detected. Executing initial clone to {ROOT_DIR}...")
    !git clone https://github.com/Geauga/Proteina-Complexa-coLab {ROOT_DIR}
else:
    print(f">>> Source repository already exists at {ROOT_DIR}. Skipping clone step.")

%cd {ROOT_DIR}

def setup_environment():
    print(">>> Initializing installation and environment configuration...")

    # 1. Base Installation
    !pip install uv
    !uv pip install --system --index-strategy unsafe-best-match -e .

    # 2. Consolidated Dependency Installation
    !uv pip install --system git+https://github.com/RosettaCommons/atomworks.git
    !uv pip install --system torch-scatter --no-build-isolation
    !uv pip install --system dm-haiku openbabel-wheel graphein e3nn

    # 3. Deploy Foldseek Binary
    if not os.path.exists(".venv/bin/foldseek"):
        !mkdir -p .venv/bin
        !wget -q -nc https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
        !tar xzf foldseek-linux-avx2.tar.gz
        !mv foldseek/bin/foldseek .venv/bin/ && chmod +x .venv/bin/foldseek
        !rm -rf foldseek-linux-avx2.tar.gz foldseek/

    # 4. Create sc Script Mock
    !mkdir -p env/docker/internal/
    !echo -e '#!/bin/bash\necho "0.000"' > env/docker/internal/sc && chmod +x env/docker/internal/sc

    # 5. Initialization and Model Download
    !complexa init uv
    !source env.sh && complexa download --complexa-all
    !source env.sh && complexa download --all

    # 6. Path Correction (.env)
    code_path = ROOT_DIR + "/"
    data_path = os.path.join(ROOT_DIR, "assets/")
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE, "r") as f: lines = f.readlines()
        with open(ENV_FILE, "w") as f:
            for line in lines:
                if line.startswith("LOCAL_CODE_PATH="): f.write(f"LOCAL_CODE_PATH={code_path}\n")
                elif line.startswith("LOCAL_DATA_PATH="): f.write(f"LOCAL_DATA_PATH={data_path}\n")
                else: f.write(line)

    # 7. Task Completion: Conditional Backup (Reads enable_cache_backup from Cell 2a)
    if enable_cache_backup:
        if not os.path.exists(UV_CACHE_TAR):
            print(">>> User opted in for cache backup. Extracting and backing up underlying C++ build cache to Drive...")
            !mkdir -p {BACKUP_DIR}
            !mkdir -p ~/.cache/uv
            !cd ~ && tar -cf {UV_CACHE_TAR} .cache/uv
            print(">>> Cache backup complete. Future reboots will achieve rapid recovery.")
        else:
            print(">>> Existing cache backup detected in Drive. Skipping redundant packaging.")
    else:
        print(">>> Cache backup bypassed per user configuration.")

# ====================================================================
# --- Main Logic: Inject cache first, then execute configuration ---
# ====================================================================
if os.path.exists(UV_CACHE_TAR):
    print(">>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...")
    !mkdir -p ~/.cache
    !tar -xf {UV_CACHE_TAR} -C ~
    print(">>> Cache injection complete. Preparing environment reconstruction...")
    setup_environment()
else:
    print(">>> No cache detected. Commencing initial heavy compilation (please wait)...")
    setup_environment()

print(">>> Environment ready. Initiating final validation...")
!source env.sh && complexa validate design configs/search_binder_local_pipeline.yaml

>>> Source repository already exists at /content/drive/MyDrive/Proteina-Complexa. Skipping clone step.
/content/drive/MyDrive/Proteina-Complexa
>>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...
>>> Cache injection complete. Preparing environment reconstruction...
>>> Initializing installation and environment configuration...
Using Python 3.12.13 environment at: /usr
Resolved 138 packages in 2.23s
Prepared 1 package in 824ms
Uninstalled 5 packages in 169ms
Installed 5 packages in 26ms
 - biotite==1.4.0
 + biotite==0.41.2
 - numpy==2.4.4
 + numpy==1.26.4
 ~ proteinfoundation==1.0.0 (from file:///content/drive/MyDrive/Proteina-Complexa)
 - scipy==1.17.1
 + scipy==1.12.0
 - wandb==0.25.1
 + wandb==0.23.1
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 1.79s
Uninstalled 2 packages in 22ms
Installed 2 packages in 27ms
 - biotite==0.41.2
 + biotite==1.4.0
 - scipy==1.12.0
 + scipy==1.17.1
Using Python 3.12.13 environment at: /usr
Checked 1 package 

In [5]:
# Cell_3_Repository_Clone_and_JAX_Patch.py
# Requirement: Detect and clone the Proteina-Complexa repository if absent. Upgrade core machine learning libraries (JAX, Flax, Haiku), and dynamically patch deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.

# ==========================================
# --- Repository Initialization & Environment Patching ---
# ==========================================
import os
import sys
import subprocess
from pathlib import Path

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")


# Align Paths
BASE_DIR = Path(ROOT_DIR)
COLABDESIGN_DIR = BASE_DIR / 'community_models/colabdesign'

def run_command(command_list):
    print(f"🚀 Executing: {' '.join(command_list)[:80]}...")
    process = subprocess.Popen(command_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout: print(line, end='', flush=True)
    process.wait()

# 1. Upgrade core dependencies (tailored for GPU environment)
print("\n[Stage 1/3] Upgrading JAX and core ecosystem libraries...")
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "jax[cuda12]", "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"])
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "flax", "dm-haiku", "chex", "optax"])

# 2. Patch JAX underlying NumPy compatibility (for Python 3.12+)
print("\n[Stage 2/3] Patching JAX and NumPy conflicts...")
literals_path = Path('/usr/local/lib/python3.12/dist-packages/jax/_src/literals.py')
if literals_path.exists():
    code = literals_path.read_text()
    if ", copy=copy" in code:
        literals_path.write_text(code.replace(", copy=copy", ""))
        print("✅ JAX copy=copy patch applied successfully!")

# 3. Deep patching for ColabDesign source code
print("\n[Stage 3/3] Patching ColabDesign source code compatibility...")
if COLABDESIGN_DIR.exists():
    for filepath in COLABDESIGN_DIR.rglob('*.py'):
        code = filepath.read_text()
        new_code = code.replace('jax.lib.xla_bridge.get_backend()', 'jax.extend.backend.get_backend()') \
                       .replace('.live_buffers()', '.live_arrays()') \
                       .replace('jax.tree_map', 'jax.tree_util.tree_map') \
                       .replace('jax.tree_flatten', 'jax.tree_util.tree_flatten') \
                       .replace('jax.tree_unflatten', 'jax.tree_util.tree_unflatten') \
                       .replace('jax.tree_leaves', 'jax.tree_util.tree_leaves')
        if '@jax.util.wraps' in new_code:
            new_code = new_code.replace('@jax.util.wraps(fun, docstr=docstr)', '@functools.wraps(fun)') \
                               .replace('@jax.util.wraps(fun)', '@functools.wraps(fun)')
            if 'import functools' not in new_code: new_code = 'import functools\n' + new_code
        if new_code != code: filepath.write_text(new_code)
    print("✅ ColabDesign patching completed.")
else:
    print("⚠️ ColabDesign directory not found. Skipping Stage 3 patching.")

print("\n✨ Environment patching is complete. You may now run the evaluation cells below.")

# Purpose: Integrate repository cloning with machine learning dependency upgrades. Patches deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.
# Upstream Code: User provided combined logic for Git cloning and JAX/ColabDesign patching.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:42 EDT.
# Changed Lines:
# * Consolidated duplicate imports and harmonized base directory variables natively.
# * Added an `else` safeguard block for Stage 3 in case the ColabDesign directory is absent at runtime.


[Stage 1/3] Upgrading JAX and core ecosystem libraries...
🚀 Executing: /usr/bin/python3 -m pip install --upgrade jax[cuda12] -f https://storage.googlea...
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphein 1.7.8 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
proteinfoundation 1.0.0 requires scipy==1.12.0, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have

# Section 2: Target Specification and Structural Pre-processing

In [6]:
# Cell_4b PDB Processing & YAML Configuration Generation



import os, json, yaml, shutil, time, sys
from pathlib import Path

print("🔍 Initializing Target Processing Engine...")

# ---------------------------------------------------------
# 1. Load Persistent Configuration from Google Drive
# ---------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

SETTING_FILE = Path('/content/drive/MyDrive/Proteina-Complexa/internal_settings.json')
if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found.")
    print("💡 Action: Please scroll up to Section 0, complete the form, and run the Global Dashboard cell first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

# Extract essential variables dynamically
pdb_file = cfg['pdb_file']
task_name = cfg['task_name']
target_chains = cfg['target_chains']
hotspots_input = str(cfg['hotspots_input'])
binder_length_min = int(cfg['binder_length_min'])
binder_length_max = int(cfg['binder_length_max'])

print(f"✅ Loaded persistent settings for task: [{task_name}]")

# ---------------------------------------------------------
# 2. Path Setup & Verification
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGETS_INPUT_DIR = BASE_DIR / 'targets'
RUN_DATA_DIR = BASE_DIR / 'assets' / 'target_data' / task_name
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

TARGETS_INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)

raw_pdb_path = TARGETS_INPUT_DIR / pdb_file
fixed_pdb_path = RUN_DATA_DIR / f"{task_name}_fixed.pdb"

if not raw_pdb_path.exists():
    print(f"❌ Error: Required PDB file '{pdb_file}' not found in '{TARGETS_INPUT_DIR}'.")
    sys.exit(1)

# ---------------------------------------------------------
# 3. PDB Parsing & Missing Gap Stitching
# ---------------------------------------------------------
print(f"⚙️ Parsing raw structure '{pdb_file}' and stitching missing residues...")

valid_res_nums = set()
with open(raw_pdb_path, 'r') as f:
    for line in f:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA' and line[21] == target_chains:
            try: valid_res_nums.add(int(line[22:26].strip()))
            except: continue

with open(raw_pdb_path, 'r') as f_in, open(fixed_pdb_path, 'w') as f_out:
    for line in f_in:
        if line.startswith('ATOM') and line[21] == target_chains:
            try:
                if int(line[22:26].strip()) in valid_res_nums: f_out.write(line)
            except: pass
        elif line.startswith('TER'): f_out.write(line)

res_nums = sorted(list(valid_res_nums))
ranges = []
if res_nums:
    start = prev = res_nums[0]
    for n in res_nums[1:]:
        if n == prev + 1: prev = n
        else:
            ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")
            start = prev = n
    ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")

# ---------------------------------------------------------
# 4. YAML Configuration Generation
# ---------------------------------------------------------
hotspots_list = [int(x.strip()) for x in hotspots_input.split(',') if x.strip().isdigit()]

yaml_data = {'target_dict_cfg': {
    task_name: {
        'target_input': ','.join(ranges),
        'binder_length': [binder_length_min, binder_length_max],
        'target_chains': [target_chains],
        'pdb_id': task_name,
        'source': "PDB",
        'target_filename': task_name + "_fixed",
        'hotspot_residues': [f"{target_chains}{i}" for i in hotspots_list]
    }
}}

YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
if YAML_PATH.exists():
    backup_name = f"targets_dict_backup_{int(time.time())}.yaml"
    shutil.copy(YAML_PATH, YAML_PATH.parent / backup_name)
    print(f"💾 Existing config safely backed up as: {backup_name}")

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"🎯 SUCCESS: Target configuration generated dynamically for '{task_name}'. Ready for generation phase!")

# ==============================================================================
# Update Log:
# Date: April 2026
# Changes:
#   - Refactored to completely decouple from Colab UI in-memory variables.
#   - Integrated persistent state management by reading parameters directly
#     from '/content/drive/MyDrive/Proteina-Complexa/internal_settings.json'.
#   - Removed obsolete interactive upload logic (now handled in Section 0).
# ==============================================================================

🔍 Initializing Target Processing Engine...
✅ Loaded persistent settings for task: [GFP_1_10]
⚙️ Parsing raw structure 'GFP_1_10.pdb' and stitching missing residues...
💾 Existing config safely backed up as: targets_dict_backup_1775179140.yaml
🎯 SUCCESS: Target configuration generated dynamically for 'GFP_1_10'. Ready for generation phase!


# Section 3: Pipeline Execution: Generative Inference

In [20]:
# Cell_5b_Standard Pipeline Execution



import os, json, sys, subprocess, datetime
from pathlib import Path

# ---------------------------------------------------------
# 1. Load Persistent Configuration (The "Memory Card")
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run Section 0 first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
num_designs = cfg['num_designs']
gen_batch_size = cfg['gen_batch_size']
run_filter = cfg['run_filter']
run_evaluate = cfg['run_evaluate']
run_analyze = cfg['run_analyze']

print(f"🚀 Initiating Generation Pipeline for Target: [{task_name}]")
print(f"📊 Designs: {num_designs} | Batch Size: {gen_batch_size}")

# ---------------------------------------------------------
# 2. Path Setup & Run Identification
# ---------------------------------------------------------
RUN_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
CURRENT_RUN_NAME = f"Run_{task_name}_{RUN_TIMESTAMP}"
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

# Save execution manifest for downstream Analyzer
RECORD_FILE = BASE_DIR / '.latest_run.txt'
with open(RECORD_FILE, 'w') as f:
    f.write(f"{CURRENT_RUN_NAME}|{task_name}")

print(f"🆔 Run Identifier: {CURRENT_RUN_NAME}")

# ---------------------------------------------------------
# 3. Native Complexa Subprocess Execution
# ---------------------------------------------------------
def run_command_with_env(command_str):
    env = os.environ.copy()
    # Apply strict native environment flags
    env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

    process = subprocess.Popen(
        f"source env.sh && {command_str}",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        shell=True,
        executable='/bin/bash',
        cwd=str(BASE_DIR),
        bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    return process.returncode

# Construct the exact native command string with dynamic JSON overrides
cmd_str = (
    f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={CURRENT_RUN_NAME} "
    f"++generation.task_name={task_name} ++generation.num_designs={num_designs} "
    f"++generation.dataloader.batch_size={gen_batch_size} ++generation.search.max_batch_size={gen_batch_size} "
    f"++generation.dataloader.dataset.conditional_features.0.pdb_path={FIXED_PDB_PATH} "
    f"++run_filter={run_filter} ++run_evaluate={run_evaluate} ++run_analyze={run_analyze}"
)

print(f"\n⚙️ Executing standard pipeline via native wrapper...")
ret_code = run_command_with_env(cmd_str)

if ret_code == 0:
    print(f"\n✅ Generation complete! Proceed to the Downstream Analyzer (Cell 7) to extract top candidates.")
else:
    print(f"\n❌ Error: Pipeline crashed (Code {ret_code}). Check logs above.")


# ==============================================================================
# Update Log:
# Date: April 2026
# Changes:
#   - Restored the original native execution wrapper (`complexa design`).
#   - Removed the broken independent `python3 -m` calls.
#   - Reads pipeline parameters seamlessly from the persistent JSON memory.
# ==============================================================================

🚀 Initiating Generation Pipeline for Target: [GFP_1_10]
📊 Designs: 8 | Batch Size: 8
🆔 Run Identifier: Run_GFP_1_10_Y2026_M04_D03_H01_M48_S38

⚙️ Executing standard pipeline via native wrapper...
Complexa environment initialized for uv runtime.

  🧬 Complexa Binder Design Pipeline
  Config:       configs/search_binder_local_pipeline.yaml
  Target:       GFP_1_10
  Run:          Run_GFP_1_10_Y2026_M04_D03_H01_M48_S38
  GPUs:         1
  Steps:        generate, filter, evaluate, analyze
  🗂️  Stage logs: logs/design_pipeline_GFP_1_10_Run_GFP_1_10_Y2026_M04_D03_H01_M48_S38_Y2026_M04_D03_H01_M48_S38

────────────────────────────────────────────────────────────
  🔥💻 Step 1/4: generate
  Generate binder structures using the partially latent flow matching model
  📝 Log: logs/design_pipeline_GFP_1_10_Run_GFP_1_10_Y2026_M04_D03_H01_M48_S38_Y2026_M04_D03_H01_M48_S38/generate.log
────────────────────────────────────────────────────────────
  ✓ generate completed in 443.5s

  ─────────────────────

In [21]:
# Cell_6_Intelligent_Error_Log_Analyzer.py
# Requirement: Automatically inspect and diagnose execution logs across the entire pipeline. Retrieve the workload manifest, scan specific log outputs for each phase, and report the exact point of failure or confirm complete success.

import os
import sys
from pathlib import Path

# ==========================================
# 1. Read Task Record (Strictly Inherit Dedicated Variables)
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'
LOGS_DIR = BASE_DIR / 'logs'

if not RECORD_FILE.exists():
    print("❌ Error: Record file .latest_run.txt not found. Please execute the pipeline first.")
    sys.exit(1)

content = RECORD_FILE.read_text().strip()
TARGET_RUN_NAME, task_str = content.split('|')
TASKS = task_str.split(',')

print(f"🎯 Retrieving dedicated run code: {TARGET_RUN_NAME}")
print(f"📋 Included target tasks: {TASKS}\n")

# ==========================================
# Intelligent Log Diagnostic Tool
# ==========================================
def check_log_status(log_path):
    """Reads the log to determine success and returns (is_success, log_tail_content)"""
    if not log_path.exists():
        return False, []

    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    if not lines:
        return False, ["(Log file is empty)"]

    # Extract only the last 50 lines for analysis and display
    tail_lines = lines[-50:]
    tail_text = "".join(tail_lines)

    # Capture key flag words indicating errors
    error_keywords = ["Traceback (most recent call last):", "Error:", "[ERROR]", "OutOfMemoryError", "Exception:"]

    for kw in error_keywords:
        if kw in tail_text:
            return False, tail_lines

    # If no error flags are present, evaluate as successful
    return True, tail_lines

# ==========================================
# 2. Iteratively Diagnose Error Logs for Each Target
# ==========================================
for task in TASKS:
    print("="*70)
    print(f"🔎 Diagnosing target [ {task} ]")
    print("="*70)

    # Core matching logic: Compatible with the dynamic long-string nomenclature
    folder_pattern = f"design_pipeline_{task}_{TARGET_RUN_NAME}*"
    matching_folders = sorted(list(LOGS_DIR.glob(folder_pattern)))

    if not matching_folders:
        print(f"❌ Log folder for {task} not found. It may have been deleted or not yet executed.")
        continue

    latest_folder = matching_folders[-1]

    gen_log = latest_folder / 'generate.log'
    filter_log = latest_folder / 'filter.log'
    eval_log = latest_folder / 'evaluate.log'
    main_log = latest_folder.with_name(latest_folder.name + ".log")

    # 1. Inspect Generate Phase
    if gen_log.exists():
        gen_success, gen_lines = check_log_status(gen_log)

        if gen_success:
            print("✅ Generate (Generation Phase) : Success!")

            # 2. Inspect Filter Phase
            if filter_log.exists():
                fil_success, fil_lines = check_log_status(filter_log)
                if fil_success:
                    print("✅ Filter   (Filtering Phase) : Success!")
                else:
                    print(f"❌ Filter   (Filtering Phase) : Failed!")
                    print(f"🔥 --- Reading {latest_folder.name}/filter.log --- 🔥\n")
                    print("".join(fil_lines))
                    continue # Halt further inspection if the filtering phase fails

            # 3. Inspect Evaluate Phase
            if eval_log.exists():
                eval_success, eval_lines = check_log_status(eval_log)
                if eval_success:
                    print("✅ Evaluate (Evaluation Phase)  : Success!")
                    print("🎉 Conclusion: All executed processes for this target have passed perfectly!\n")
                else:
                    print(f"❌ Evaluate (Evaluation Phase)  : Failed!")
                    print(f"🔥 --- Reading {latest_folder.name}/evaluate.log --- 🔥\n")
                    print("".join(eval_lines))
            else:
                print("⚠️ Evaluate (Evaluation Phase)  : evaluate.log not found.")
                print("   (Note: The current task may have only completed generation and filtering; evaluation was either disabled or not yet reached.)\n")

        else:
            print(f"❌ Generate (Generation Phase) : Failed!")
            print(f"🔥 --- Reading {latest_folder.name}/generate.log --- 🔥\n")
            print("".join(gen_lines))

    # If the internal generate.log is missing, attempt to read the outer wrapper's main log
    elif main_log.exists():
        print(f"⚠️ Step-specific logs not found. Attempting to read main log: {main_log.name} \n")
        with open(main_log, 'r', encoding='utf-8') as f:
            print("".join(f.readlines()[-40:]))
    else:
        print(f"❌ No log files found in directory {latest_folder.name}!\n")

# Purpose: Automatically inspect and diagnose execution logs across the entire pipeline to identify and report points of failure.
# Upstream Code: Original Cell 5 (Log Analyzer) from the Proteina_Complexa_coLab.ipynb.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:00 EDT.
# Changed Lines:
# * Renumbered to Cell 6 to maintain sequential logical flow.


🎯 Retrieving dedicated run code: Run_GFP_1_10_Y2026_M04_D03_H01_M48_S38
📋 Included target tasks: ['GFP_1_10']

🔎 Diagnosing target [ GFP_1_10 ]
✅ Generate (Generation Phase) : Success!
✅ Filter   (Filtering Phase) : Success!
✅ Evaluate (Evaluation Phase)  : Success!
🎉 Conclusion: All executed processes for this target have passed perfectly!



In [22]:
# Cell_7_Deep_Result_Scanner.py
# Requirement: Dynamically retrieve and parse inference results based on the workload manifest.
# Read the target_threshold dynamically from the persistent configuration file to replace hardcoded values.
# Strictly rank generated designs using min_ipSAE, highlight hits, and display auxiliary metrics.

import os
import sys
import csv
import json
from pathlib import Path

# ==========================================
# 1. Load Persistent Configuration & Run Record
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'
SETTING_FILE = BASE_DIR / 'internal_settings.json'

# Dynamically load configuration to avoid hardcoded parameters
if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run the setup block first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

# Extract dynamic threshold, fallback to 0.80 if key is missing
TARGET_THRESHOLD = float(cfg.get('target_threshold', 0.80))

if not RECORD_FILE.exists():
    print("❌ Error: Record file .latest_run.txt not found. Please execute the generation pipeline first.")
    sys.exit(1)

content = RECORD_FILE.read_text().strip()
TARGET_RUN_NAME, task_str = content.split('|')
TASKS = task_str.split(',')

print(f"🎯 Retrieving dedicated run code: {TARGET_RUN_NAME}")
print(f"📋 Included target tasks: {TASKS}")
print(f"⚙️ Active screening threshold: {TARGET_THRESHOLD}\n")

# ==========================================
# 2. Iteratively Scan and Parse Results for Each Target
# ==========================================
for task in TASKS:
    print("\n" + "=" * 85)
    print(f"  Target List: [ {task} ]")
    print("=" * 85)

    # Use wildcard matching to accommodate various suffixes added by the underlying framework
    inference_base = BASE_DIR / 'inference'
    folder_pattern = f"*{task}_{TARGET_RUN_NAME}*"
    matching_folders = sorted(list(inference_base.glob(folder_pattern)))

    if not matching_folders:
        print(f"❌ Inference result folder for {task} not found. Generation may be incomplete or an error occurred.")
        continue

    # Retrieve the most recent matching folder
    INFERENCE_DIR = matching_folders[-1]

    all_csv_files = list(INFERENCE_DIR.rglob('*.csv'))
    valid_csv_files = [f for f in all_csv_files if 'timing' not in f.name.lower()]

    if not valid_csv_files:
        print(f"⚠️ No valid CSV result file found in {INFERENCE_DIR.name}.")
        continue

    target_csv = max(valid_csv_files, key=lambda x: x.stat().st_size)

    with open(target_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        print("⚠️ CSV file is empty.")
        continue

    # --- Intelligent Sorting & Extraction Logic ---
    def get_val(row, col):
        try: return float(row.get(col, -999.0))
        except: return -999.0

    # Strictly sort in descending order based on af2folding_min_ipsae
    rows.sort(key=lambda x: get_val(x, 'af2folding_min_ipsae'), reverse=True)

    # --- Print Professional Evaluation Table ---
    header_format = "{:<4} | {:<12} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
    print(header_format.format("Hit", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD(Å)", "Reward"))
    print("-" * 85)

    hits_found = 0
    # Display the top 10 designs from this single run
    for i, row in enumerate(rows[:10]):
        # Correct ID: If design_id is empty, fallback to index naming
        d_id = row.get('design_id', '').strip()
        if not d_id: d_id = f"Design_{row.get('pdb_index', i)}"

        # Extract precise metrics
        ipsae = get_val(row, 'af2folding_min_ipsae')
        ptm = get_val(row, 'af2folding_ptm_log')
        iptm = get_val(row, 'af2folding_i_ptm_log')
        plddt = get_val(row, 'af2folding_plddt')
        rmsd = get_val(row, 'af2folding_rmsd')
        reward = get_val(row, 'total_reward')

        # Mark hits visually
        is_hit = ipsae >= TARGET_THRESHOLD
        hit_marker = "🌟" if is_hit else "  "
        if is_hit: hits_found += 1

        # Format and print the row safely
        ipsae_str = f"{ipsae:.4f}" if ipsae != -999.0 else "N/A"
        ptm_str = f"{ptm:.4f}" if ptm != -999.0 else "N/A"
        iptm_str = f"{iptm:.4f}" if iptm != -999.0 else "N/A"
        plddt_str = f"{plddt:.2f}" if plddt != -999.0 else "N/A"
        rmsd_str = f"{rmsd:.2f}" if rmsd != -999.0 else "N/A"
        reward_str = f"{reward:.4f}" if reward != -999.0 else "N/A"

        row_format = "{:<4} | {:<12} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
        print(row_format.format(hit_marker, d_id[:12], ipsae_str, ptm_str, iptm_str, plddt_str, rmsd_str, reward_str))

    print(f"\n✅ Scan Summary: Found {hits_found} 'Hits' (ipSAE >= {TARGET_THRESHOLD}) in the top results.")
    print(f"📂 Source Data Path: {INFERENCE_DIR.relative_to(BASE_DIR)}")

print("\n🎉 Single-Target Leaderboard scan complete. The top-scoring designs are identified above.")

# Purpose: Automate the retrieval and parsing of generated protein designs. Dynamically reads user-defined thresholds instead of hardcoding.
# Upstream Code: Pipeline generation output (Cell 6) and global configuration JSON.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-02 21:28:02 EDT.
# Changed Lines:
# - Line 9: Added `import json`.
# - Lines 18-25: Added logic to locate and parse `internal_settings.json` to dynamically extract the `target_threshold` variable.
# - Line 36: Added a console printout `print(f"⚙️ Active screening threshold: {TARGET_THRESHOLD}\n")` to verify the parsed configuration parameters.

🎯 Retrieving dedicated run code: Run_GFP_1_10_Y2026_M04_D03_H01_M48_S38
📋 Included target tasks: ['GFP_1_10']
⚙️ Active screening threshold: 0.8


  Target List: [ GFP_1_10 ]
Hit  | Design_ID    | ipSAE    | pTM      | iPTM     | pLDDT    | RMSD(Å)  | Reward  
-------------------------------------------------------------------------------------
     | Design_1     | 0.3128   | 0.9181   | 0.9169   | 0.07     | 0.37     | -0.1299 
     | Design_0     | 0.2671   | 0.9147   | 0.8813   | 0.12     | 1.91     | -0.1769 
     | Design_4     | 0.2666   | 0.9136   | 0.8801   | 0.12     | 1.70     | -0.1672 
     | Design_7     | 0.2526   | 0.9169   | 0.8690   | 0.12     | 2.99     | -0.1712 
     | Design_6     | 0.2057   | 0.9112   | 0.8270   | 0.17     | 2.50     | -0.2117 
     | Design_5     | 0.1161   | 0.9088   | 0.7502   | 0.27     | 30.44    | -0.2940 
     | Design_3     | 0.0551   | 0.8930   | 0.7369   | 0.35     | 23.79    | -0.3374 
     | Design_2     | 0.0000   | 0.8443   | 0.0803 

In [10]:
# Cell_8_Batch_Packaging_and_Download.py
# Requirement: Consolidate generated PDB and CSV files across multiple targets into a single staging directory based on the workload manifest, compress them into a unified ZIP archive, and trigger an automatic browser download.

import os
import shutil
import sys
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    print("⚠️ Warning: Not executing within a Colab environment. The system will only compress the files without triggering a browser download.")

# ==========================================
# 1. Read Task Record (Strictly Inherit Dedicated Variables)
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'

if not RECORD_FILE.exists():
    print("❌ Error: Record file not found. Unable to determine tasks for packaging. Please execute the generation pipeline first.")
    sys.exit(1)

content = RECORD_FILE.read_text().strip()
TARGET_RUN_NAME, task_str = content.split('|')
TASKS = task_str.split(',')

print(f"🎯 Preparing to package dedicated run code: {TARGET_RUN_NAME}")
print(f"📋 Included tasks: {TASKS}\n")

# ==========================================
# 2. Assemble Staging Directory
# ==========================================
# Create a temporary directory to consolidate results from all targets
ARCHIVE_STAGING_DIR = BASE_DIR / f'Batch_Results_Staging_{TARGET_RUN_NAME}'
ARCHIVE_STAGING_DIR.mkdir(parents=True, exist_ok=True)

total_pdbs = 0
total_csvs = 0

inference_base = BASE_DIR / 'inference'

for task in TASKS:
    # Core Modification: Utilize wildcard matching to automatically track extended suffix directories generated by the underlying framework
    folder_pattern = f"*{task}_{TARGET_RUN_NAME}*"
    matching_folders = sorted(list(inference_base.glob(folder_pattern)))

    if matching_folders:
        # Retrieve the most recent result directory
        INFERENCE_DIR = matching_folders[-1]

        pdbs = list(INFERENCE_DIR.rglob('*.pdb'))
        csvs = list(INFERENCE_DIR.rglob('*.csv'))
        total_pdbs += len(pdbs)
        total_csvs += len(csvs)

        # Copy the entire result directory for this target into the staging area, naming the subdirectory after the task
        dest_dir = ARCHIVE_STAGING_DIR / task
        if dest_dir.exists():
            shutil.rmtree(dest_dir) # Clear pre-existing remnants if any
        shutil.copytree(INFERENCE_DIR, dest_dir)

        print(f"✅ Consolidated [ {task} ] into staging (includes {len(pdbs)} PDBs, {len(csvs)} CSVs)")
    else:
        print(f"⚠️ Result directory for target [ {task} ] not found (Pattern: {folder_pattern}). Skipped.")

# ==========================================
# 3. Compression and Download
# ==========================================
if total_pdbs > 0:
    ZIP_FILE_PATH = BASE_DIR / f'All_Results_{TARGET_RUN_NAME}_Full'
    print(f"\n📦 Compressing all contents into a single archive. Please wait...")

    # Execute compression
    shutil.make_archive(str(ZIP_FILE_PATH), 'zip', ARCHIVE_STAGING_DIR)
    final_zip = str(ZIP_FILE_PATH) + '.zip'
    print(f"🎉 Compression complete! Drive backup saved at: {final_zip}")

    # Clean up the temporary staging directory to maintain storage hygiene
    shutil.rmtree(ARCHIVE_STAGING_DIR)

    # Trigger browser download
    print("⬇️ Requesting browser download. Please monitor for download prompts (check browser popup blockers if none appear)...")
    try:
        files.download(final_zip)
    except NameError:
        pass
else:
    print("\n❌ All directories are empty. No files found to package. Aborting.")
    # Clean up the empty staging directory
    if ARCHIVE_STAGING_DIR.exists():
        shutil.rmtree(ARCHIVE_STAGING_DIR)

# Purpose: Consolidate generated structures and metrics into a centralized staging area, generate an archive for local analysis, and clear temporary files to maintain Google Drive storage hygiene.
# Upstream Code: Original Cell 4 (Batch Packaging and Download) from the Proteina_Complexa_coLab.ipynb.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:10 EDT.
# Changed Lines:
# * Renumbered to Cell 8 to maintain sequential logical flow.


🎯 Preparing to package dedicated run code: Run_GFP_1_10_Y2026_M04_D02_H23_M58_S31
📋 Included tasks: ['GFP_1_10']

✅ Consolidated [ GFP_1_10 ] into staging (includes 8 PDBs, 4 CSVs)

📦 Compressing all contents into a single archive. Please wait...
🎉 Compression complete! Drive backup saved at: /content/drive/MyDrive/Proteina-Complexa/All_Results_Run_GFP_1_10_Y2026_M04_D02_H23_M58_S31_Full.zip
⬇️ Requesting browser download. Please monitor for download prompts (check browser popup blockers if none appear)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Section 4: High-Throughput Autonomous Screening

In [ ]:
# Cell_9b_Autonomous_Generation_Loop_Execution.py

import os, json, sys, time, subprocess
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. Load Persistent Configuration
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run Section 0 first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_threshold = float(cfg['target_threshold'])
target_success_count = int(cfg['target_success_count'])
designs_per_loop = int(cfg['designs_per_loop'])
max_iterations = int(cfg['max_iterations'])
batch_size = int(cfg['batch_size'])

print(f"🤖 Auto-Pilot Activated for Target: [{task_name}]")
print(f"🎯 Mission: Collect {target_success_count} hits exceeding threshold score {target_threshold}.")
print(f"🔄 Loop Settings: {designs_per_loop} designs per cycle (Max {max_iterations} cycles).")

# ---------------------------------------------------------
# 2. Tracking Variables Setup
# ---------------------------------------------------------
current_iteration = 0
collected_hits = 0
SCREENING_RESULTS_DIR = Path(f'/content/drive/MyDrive/Proteina-Complexa/screening_results/{task_name}')
SCREENING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
master_log_file = SCREENING_RESULTS_DIR / "autopilot_master_log.csv"

# ---------------------------------------------------------
# 3. High-Throughput Engine Loop
# ---------------------------------------------------------
while current_iteration < max_iterations and collected_hits < target_success_count:
    current_iteration += 1
    print(f"\n=======================================================")
    print(f"🔄 Auto-Pilot Cycle [{current_iteration}/{max_iterations}] Initiated...")
    print(f"=======================================================")

    # [BUG FIXED]: Replaced isolated python3 -m calls with native complexa wrapper
    FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

    env = os.environ.copy()
    env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml "
        f"++generation.task_name={task_name} ++generation.num_designs={designs_per_loop} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={FIXED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=False ++run_analyze=False"
    )

    # Run Generation and Filtering together via wrapper
    subprocess.run(
        f"source env.sh && {cmd_str}",
        env=env,
        shell=True,
        executable='/bin/bash',
        cwd=str(BASE_DIR),
        check=False
    )

    # Example logic to parse outputs (Unchanged)
    cycle_csv_path = Path(f"/content/drive/MyDrive/Proteina-Complexa/inference/{task_name}/current_batch_scores.csv")

    if cycle_csv_path.exists():
        try:
            df = pd.read_csv(cycle_csv_path)
            # Assuming 'ipSAE' is the column name for the score
            if 'ipSAE' in df.columns:
                hits_in_cycle = df[df['ipSAE'] >= target_threshold].shape[0]
                collected_hits += hits_in_cycle

                # Append to master log
                df[df['ipSAE'] >= target_threshold].to_csv(master_log_file, mode='a', header=not master_log_file.exists(), index=False)
                print(f"✨ Found {hits_in_cycle} valid hits in cycle {current_iteration}!")
        except Exception as e:
            print(f"⚠️ Warning: Could not parse cycle results: {e}")

    print(f"📊 Progress: {collected_hits} / {target_success_count} hits collected so far.")

    # VRAM Cleanup
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 VRAM cleared for next cycle.")

    time.sleep(2) # Brief cooldown

# ---------------------------------------------------------
# 4. Auto-Pilot Completion Summary
# ---------------------------------------------------------
print(f"\n🛑 Auto-Pilot execution finished.")
if collected_hits >= target_success_count:
    print(f"🏆 SUCCESS: Target hit count reached! {collected_hits} high-quality binders designed.")
else:
    print(f"⚠️ REACHED MAX ITERATIONS: Collected {collected_hits} hits after {max_iterations} cycles.")

print(f"📂 Master Log saved at: {master_log_file}")

# ==============================================================================
# Update Log:
# Date: April 2026
# Changes:
#   - Converted to pure backend engine reading from Google Drive config JSON.
#   - Retrieves 'target_success_count', 'target_threshold', etc., enabling
#     seamless unattended high-throughput screening without UI memory reliance.
#   - [Hotfix] Integrated 'complexa design' native wrapper to resolve Hydra config issues.
# ==============================================================================

🤖 Auto-Pilot Activated for Target: [GFP_1_10]
🎯 Mission: Collect 10 hits exceeding threshold score 0.8.
🔄 Loop Settings: 24 designs per cycle (Max 50 cycles).

🔄 Auto-Pilot Cycle [1/50] Initiated...
🆔 Cycle Identifier: AutoPilot_GFP_1_10_Iter1_Y2026_M04_D03_H02_M10_S43
⚙️ Running generation and filtering engine...
Complexa environment initialized for uv runtime.

  🧬 Complexa Binder Design Pipeline
  Config:       configs/search_binder_local_pipeline.yaml
  Target:       GFP_1_10
  Run:          AutoPilot_GFP_1_10_Iter1_Y2026_M04_D03_H02_M10_S43
  GPUs:         1
  Steps:        generate, filter, evaluate, analyze
  🗂️  Stage logs: logs/design_pipeline_GFP_1_10_AutoPilot_GFP_1_10_Iter1_Y2026_M04_D03_H02_M10_S43_Y2026_M04_D03_H02_M10_S43

────────────────────────────────────────────────────────────
  🔥💻 Step 1/4: generate
  Generate binder structures using the partially latent flow matching model
  📝 Log: logs/design_pipeline_GFP_1_10_AutoPilot_GFP_1_10_Iter1_Y2026_M04_D03_H02_M10_S43_Y

# Section 5: Modular Execution: Decoupled Workflow

In [ ]:
# Cell_2a_Pure_Generation_and_Filtering.py
# Requirement: Execute the pure generation and filtering phases of the pipeline by directly invoking the underlying Python modules. This bypasses the main complexa wrapper to prevent cascaded execution of evaluation stages, while inheriting dynamic directories and managing VRAM.

import os
import subprocess
import sys
import gc
import torch
from pathlib import Path

# ==========================================
# --- User Runtime Parameters ---
# ==========================================
TOTAL_DESIGNS = 16  # Total number of designs to generate
BATCH_SIZE = 16     # Parallel batch size for the generation phase (can be set to a large value)
BATCH_SIZE = min(BATCH_SIZE, TOTAL_DESIGNS)

# ==========================================
# 1. Path Verification and Task Inheritance
# (Strictly inherits the timestamp variable from Cell 1)
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'

if not RECORD_FILE.exists():
    print("❌ Error: Workload manifest .latest_run.txt not found. Please run Cell 1 first.")
    sys.exit(1)

# Inherit RUN_NAME and task list from .latest_run.txt
record_data = RECORD_FILE.read_text().strip().split('|')
if len(record_data) == 2:
    TARGET_RUN_NAME = record_data[0] # Directly acquire the dedicated run code generated in Cell 1
    TASKS = record_data[1].split(',')
else:
    print("❌ Error: .latest_run.txt format is corrupted.")
    sys.exit(1)

def run_command_with_env(command_str):
    # Physically clear VRAM
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    env = os.environ.copy()
    env.update({
        'HYDRA_FULL_ERROR': '1',
        'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
        'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
        'TF_FORCE_GPU_ALLOW_GROWTH': 'true'
    })

    full_command = f"source env.sh && {command_str}"
    process = subprocess.Popen(
        full_command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()

print("\n" + "="*50)
print(f"🚀 Initiating Pure Generation Task (Generate & Filter Only)")
print(f"📋 Task List: {TASKS}")
print(f"🎯 Dedicated Run Code: {TARGET_RUN_NAME}")
print("="*50 + "\n")

for task_name in TASKS:
    # Core Modification: Locate the cleaned PDB in the dedicated directory created in Cell 1
    fixed_pdb_path = BASE_DIR / f"assets/target_data/{TARGET_RUN_NAME}/{task_name}_fixed.pdb"

    if not fixed_pdb_path.exists():
        print(f"⚠️ Skipping {task_name}: Cleaned PDB not found at {fixed_pdb_path}")
        continue

    print(f"\n🔥 Generating: {task_name} ...")

    # Base arguments (passing the unified TARGET_RUN_NAME to the underlying framework)
    base_args = (
        "--config-path configs --config-name search_binder_local_pipeline ++job_id=0 "
        f"++run_name={TARGET_RUN_NAME} ++generation.task_name={task_name} "
        f"++generation.num_designs={TOTAL_DESIGNS} "
        f"++generation.dataloader.batch_size={BATCH_SIZE} "
        f"++generation.search.max_batch_size={BATCH_SIZE} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={fixed_pdb_path}"
    )

    # Directly invoke underlying python modules to bypass the main wrapper and prevent cascaded execution
    print("   [1/2] Executing Generation Module (Generate)...")
    run_command_with_env(f"python3 -m proteinfoundation.generate {base_args} ++gen_njobs=1")

    print("   [2/2] Executing Filtering Module (Filter)...")
    run_command_with_env(f"python3 -m proteinfoundation.filter {base_args}")

print("\n🎉 All molecular generation phases have completely finished! VRAM has been released.")

# Purpose: Execute the structure generation and initial filtering stages exclusively. By directly targeting the foundational python modules, it bypasses the outer framework wrapper, preventing automatic transition into unstable evaluation stages while ensuring proper VRAM management.
# Upstream Code: User-provided Colab cell labeled "Cell 2a: Pure Generation and Filtering Phase".
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:20 EDT.

In [ ]:
# Cell_2b_Automated_Evaluation_and_Analysis.py
# Requirement: Independently execute the evaluation and analysis modules of the protein pipeline. Dynamically inherit or manually specify the run code to locate generated structures, assess them using predictive oracles (e.g., AF2), and compile scoring reports while managing VRAM limits.

import os
import subprocess
import sys
import yaml
import gc
import torch
from pathlib import Path

# ==========================================
# --- User Configuration ---
# ==========================================
# To manually evaluate a specific run, enter the complete run code; leave blank to automatically read the latest record.
MANUAL_RUN_NAME = ""
# VRAM optimization: Adjust batch size based on available GPU memory (e.g., 8 for 80GB VRAM, 2 for 16GB VRAM).
BATCH_SIZE_EVAL = 8

# ==========================================
# 1. Core Integration Logic (Strict Inheritance or Manual Override)
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

if MANUAL_RUN_NAME.strip():
    TARGET_RUN_NAME = MANUAL_RUN_NAME.strip()
    # In manual mode, attempt to read target tasks from the current YAML configuration
    if YAML_PATH.exists():
        with open(YAML_PATH, 'r') as f:
            TASKS = list(yaml.safe_load(f).get('target_dict_cfg', {}).keys())
    else:
        print("❌ Error: Manual mode requires targets_dict.yaml to confirm the task list, but the file does not exist.")
        sys.exit(1)
    print(f"🎯 [Manual Mode] Preparing evaluation for: {TARGET_RUN_NAME}")
    print(f"📋 Included tasks: {TASKS}")
elif RECORD_FILE.exists():
    content = RECORD_FILE.read_text().strip()
    TARGET_RUN_NAME, task_str = content.split('|')
    TASKS = task_str.split(',')
    print(f"🎯 [Auto Mode] Detected latest dedicated run code: {TARGET_RUN_NAME}")
    print(f"📋 Included tasks: {TASKS}")
else:
    print("❌ Error: Generation record not found. Please execute the generation cell first or specify MANUAL_RUN_NAME.")
    sys.exit(1)

def run_PF_command(command_str):
    """Execute command with environment sourcing and VRAM cleanup mechanisms."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    env = os.environ.copy()
    env.update({
        'HYDRA_FULL_ERROR': '1',
        'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
        'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
        'TF_FORCE_GPU_ALLOW_GROWTH': 'true'
    })

    full_command = f"source env.sh && {command_str}"

    process = subprocess.Popen(
        full_command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()

# ==========================================
# 2. Iterative Evaluation and Analysis Loop
# ==========================================
for task in TASKS:
    print("\n" + "="*60)
    print(f"📊 Processing scoring task: {task}")
    print("="*60)

    # Core Modification: Directly read from the dedicated directory established in Cell 1 to avoid redundant file duplication.
    fixed_pdb_path = BASE_DIR / f"assets/target_data/{TARGET_RUN_NAME}/{task}_fixed.pdb"

    if not fixed_pdb_path.exists():
        print(f"⚠️ Warning: {fixed_pdb_path} not found. The evaluation module may encounter errors.")

    # Construct base parameter string
    base_args = (
        "--config-path configs --config-name search_binder_local_pipeline ++job_id=0 "
        f"++generation.task_name={task} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={fixed_pdb_path} "
        f"++generation.dataloader.batch_size={BATCH_SIZE_EVAL} "
        f"++generation.search.max_batch_size={BATCH_SIZE_EVAL} "
        f"++run_name={TARGET_RUN_NAME} "
    )

    # Execute Evaluation Phase
    print(f"\n[Step 1] Calculating Scores (Evaluate) - {task}...")
    run_PF_command(f"python3 -m proteinfoundation.evaluate {base_args} ++run_evaluate=True ++eval_njobs=1")

    # Execute Analysis Phase
    print(f"\n[Step 2] Generating Reports (Analyze) - {task}...")
    run_PF_command(f"python3 -m proteinfoundation.analyze {base_args} ++run_analyze=True")

print("\n🎉 All evaluation and analysis tasks have completed! Please use the subsequent scanning or packaging cells to extract the results.")

# Purpose: Automate the sequential execution of structure evaluation (e.g., ColabDesign/AF2 structural prediction) and analysis reporting. Ensures clean separation from generation stages while dynamically tracking task records for accurate file targeting.
# Upstream Code: User-provided Colab cell "Cell 2.2: Multi-task Automated Evaluation and Analysis".
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:25 EDT.
# Changed Lines:
# * Renamed internally to Cell_2b for continuous structural flow.